In [1]:
# This cell only looks; it changes nothing. Run it and read the last line.

import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn', 'joblib']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.11.9
Folder  : P02-first-model
  numpy          ok
  pandas         ok
  sklearn        ok
  joblib         ok

All good. You can carry on to Step 1.


In [2]:
# The delivery-time dataset every practical in this course uses.
# If the file is missing we build it again from the same seed, so every
# student in the room gets byte-for-byte the same 600 rows.

import csv
from pathlib import Path

import numpy as np

SEED = 42
N_ROWS = 600
DATA = Path("..") / "data" / "delivery_times.csv"


def make_delivery_csv(path=DATA):
    """Write the 600-row delivery dataset. Same formula as the lectures."""
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1,
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level",
                    "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]),
                        int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path


if not DATA.exists():
    make_delivery_csv()
    print("dataset rebuilt ->", DATA)
else:
    print("dataset found   ->", DATA)

dataset found   -> ..\data\delivery_times.csv


something went wrong


In [3]:
import logging
import sys
from pathlib import Path

WORK = Path("work")
WORK.mkdir(exist_ok=True)

logger = logging.getLogger("delivery")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()          # re-running this cell is now safe
logger.propagate = False

to_screen = logging.StreamHandler(sys.stdout)
to_screen.setFormatter(logging.Formatter(
    "%(asctime)s  %(levelname)-8s  %(name)s  %(message)s"
))
logger.addHandler(to_screen)

logger.info("logger ready")

2026-11-27 22:49:15,189  INFO      delivery  logger ready


In [4]:
logger.debug("feature vector built: [5.0, 20, 2, 0]")
logger.info("prediction served in 3 ms")
logger.warning("distance 47 km is unusually far")
logger.error("model file could not be read")
logger.critical("out of memory, shutting down")

2026-11-27 22:49:23,085  DEBUG     delivery  feature vector built: [5.0, 20, 2, 0]
2026-11-27 22:49:23,087  INFO      delivery  prediction served in 3 ms
2026-11-27 22:49:23,087  WARNING   delivery  distance 47 km is unusually far
2026-11-27 22:49:23,088  ERROR     delivery  model file could not be read
2026-11-27 22:49:23,089  CRITICAL  delivery  out of memory, shutting down


In [5]:
logger.setLevel(logging.WARNING)

print("--- threshold is now WARNING ---")
logger.debug("you will not see this")
logger.info("nor this")
logger.warning("but you will see this")
logger.error("and this")

logger.setLevel(logging.DEBUG)     # back to chatty for the lab

--- threshold is now WARNING ---
2026-11-27 22:49:26,527  WARNING   delivery  but you will see this
2026-11-27 22:49:26,529  ERROR     delivery  and this


In [6]:
import json
from datetime import datetime, timezone


class JsonFormatter(logging.Formatter):
    """Turn one log record into one line of JSON."""

    def format(self, record):
        event = {
            "time": datetime.fromtimestamp(
                record.created, timezone.utc
            ).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        # anything passed as extra={...} is added here
        for key, value in getattr(record, "fields", {}).items():
            event[key] = value
        return json.dumps(event)


def log_event(level, message, **fields):
    """Log one message with named fields attached."""
    logger.log(level, message, extra={"fields": fields})


to_screen.setFormatter(JsonFormatter())

log_event(logging.INFO, "prediction served",
          duration_ms=3, distance_km=5.0, predicted_min=42.1)

{"time": "2026-11-27T17:19:30.138801+00:00", "level": "INFO", "logger": "delivery", "message": "prediction served", "duration_ms": 3, "distance_km": 5.0, "predicted_min": 42.1}


In [7]:
LOG_FILE = WORK / "service.log"
LOG_FILE.unlink(missing_ok=True)

to_file = logging.FileHandler(LOG_FILE, encoding="utf-8")
to_file.setFormatter(JsonFormatter())
logger.addHandler(to_file)

log_event(logging.INFO, "service started", version="1.0.0")
print("logging to", LOG_FILE)

{"time": "2026-11-27T17:19:31.127356+00:00", "level": "INFO", "logger": "delivery", "message": "service started", "version": "1.0.0"}
logging to work\service.log


In [8]:
import random
import time
import uuid

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
orders = pd.read_csv(DATA)
X_train, X_test, y_train, y_test = train_test_split(
    orders[FEATURES], orders["delivery_min"],
    test_size=0.2, random_state=42,
)
model = LinearRegression().fit(X_train, y_train)

rng = random.Random(42)
to_screen.setLevel(logging.WARNING)   # keep the notebook readable

for i in range(40):
    request_id = str(uuid.uuid4())[:8]
    row = X_test.iloc[[i]]
    started = time.perf_counter()

    if i in (11, 27):                 # two deliberate failures
        log_event(logging.ERROR, "prediction failed",
                  request_id=request_id,
                  error="ValueError: traffic_level out of range",
                  duration_ms=round(
                      (time.perf_counter() - started) * 1000, 2))
        continue

    minutes = float(model.predict(row)[0])
    duration = (time.perf_counter() - started) * 1000
    if rng.random() < 0.15:           # a few slow ones
        duration += rng.uniform(200, 400)

    log_event(logging.INFO, "prediction served",
              request_id=request_id,
              distance_km=float(row["distance_km"].iloc[0]),
              traffic_level=int(row["traffic_level"].iloc[0]),
              predicted_min=round(minutes, 1),
              duration_ms=round(duration, 2))

to_screen.setLevel(logging.NOTSET)
print("40 requests logged to", LOG_FILE)
print("lines written:",
      len(LOG_FILE.read_text(encoding='utf-8').splitlines()))

{"time": "2026-11-27T17:19:33.867592+00:00", "level": "ERROR", "logger": "delivery", "message": "prediction failed", "request_id": "2d6b1f4a", "error": "ValueError: traffic_level out of range", "duration_ms": 0.0}
{"time": "2026-11-27T17:19:33.886140+00:00", "level": "ERROR", "logger": "delivery", "message": "prediction failed", "request_id": "e3d3d932", "error": "ValueError: traffic_level out of range", "duration_ms": 0.0}
40 requests logged to work\service.log
lines written: 41


In [9]:
events = [json.loads(line)
          for line in LOG_FILE.read_text(encoding="utf-8").splitlines()
          if line.strip()]

print("total events :", len(events))

errors = [e for e in events if e["level"] == "ERROR"]
print("errors       :", len(errors))

served = [e for e in events if e.get("message") == "prediction served"]
durations = [e["duration_ms"] for e in served]
print("predictions  :", len(served))
print("average ms   :", round(sum(durations) / len(durations), 1))

slow = [e for e in served if e["duration_ms"] > 200]
print("slower than 200 ms:", len(slow))
print()
print("The slowest request:")
worst = max(served, key=lambda e: e["duration_ms"])
print(f"  request_id {worst['request_id']}  "
      f"{worst['duration_ms']} ms  "
      f"{worst['distance_km']} km")

total events : 41
errors       : 2
predictions  : 38
average ms   : 56.0
slower than 200 ms: 8

The slowest request:
  request_id c456e123  362.16 ms  9.06 km


In [10]:
import hashlib

bad_order = {
    "customer_name": "Ananya Sharma",
    "phone": "9876543210",
    "address": "Flat 402, Green Park, Noida",
    "distance_km": 5.0,
    "traffic_level": 2,
}


def safe_for_logging(order):
    """Keep what helps debugging, remove what identifies a person."""
    secret = str(order.get("phone", ""))
    return {
        "customer_ref": hashlib.sha256(
            secret.encode()).hexdigest()[:12],
        "distance_km": order["distance_km"],
        "traffic_level": order["traffic_level"],
    }


print("NEVER log this:")
print("  ", bad_order)
print()
print("Log this instead:")
print("  ", safe_for_logging(bad_order))

log_event(logging.INFO, "order received",
          **safe_for_logging(bad_order))

NEVER log this:
   {'customer_name': 'Ananya Sharma', 'phone': '9876543210', 'address': 'Flat 402, Green Park, Noida', 'distance_km': 5.0, 'traffic_level': 2}

Log this instead:
   {'customer_ref': '7619ee8cea49', 'distance_km': 5.0, 'traffic_level': 2}
{"time": "2026-11-27T17:19:33.931615+00:00", "level": "INFO", "logger": "delivery", "message": "order received", "customer_ref": "7619ee8cea49", "distance_km": 5.0, "traffic_level": 2}


In [11]:
pipeline = [
    ("your code", "writes one JSON line per event to stdout"),
    ("the container runtime", "captures stdout to a file on the node"),
    ("a collector (Fluent Bit / FluentD)",
     "runs on every node and reads those files"),
    ("a store (Elasticsearch / OpenSearch)",
     "indexes the events so they can be searched"),
    ("a viewer (Kibana / Grafana)", "is where a human asks questions"),
]

for i, (part, does) in enumerate(pipeline, 1):
    print(f"  {i}. {part}")
    print(f"     {does}")

print()
print("Two rules that follow from this picture:")
print("  - never write logs to a file inside a container;")
print("    the container dies and takes the file with it")
print("  - never log secrets; that store is searchable by")
print("    everyone in the company")

  1. your code
     writes one JSON line per event to stdout
  2. the container runtime
     captures stdout to a file on the node
  3. a collector (Fluent Bit / FluentD)
     runs on every node and reads those files
  4. a store (Elasticsearch / OpenSearch)
     indexes the events so they can be searched
  5. a viewer (Kibana / Grafana)
     is where a human asks questions

Two rules that follow from this picture:
  - never write logs to a file inside a container;
    the container dies and takes the file with it
  - never log secrets; that store is searchable by
    everyone in the company


In [12]:
LEVELS = {"DEBUG": logging.DEBUG, "INFO": logging.INFO,
          "WARNING": logging.WARNING, "ERROR": logging.ERROR,
          "CRITICAL": logging.CRITICAL}

logger.setLevel(logging.WARNING)
T1_kept = [name for name, lvl in LEVELS.items() if logger.isEnabledFor(lvl)]

print("still recorded:", T1_kept)

still recorded: ['WARNING', 'ERROR', 'CRITICAL']


In [13]:
sample_record = logger.makeRecord(
    "delivery", logging.INFO, "x", 1, "prediction served", None, None
)


class ServiceFormatter(logging.Formatter):
    def format(self, record):
        event = {
            "ts": datetime.fromtimestamp(record.created, timezone.utc).isoformat(),
            "level": record.levelname,
            "service": "delivery-api",
            "msg": record.getMessage(),
            "env": "lab",
        }
        return json.dumps(event)


T2_line = json.loads(ServiceFormatter().format(sample_record))

print(T2_line)

{'ts': '2026-11-27T17:19:34.282726+00:00', 'level': 'INFO', 'service': 'delivery-api', 'msg': 'prediction served', 'env': 'lab'}


In [14]:
served_events = [e for e in events
                 if e.get("message") == "prediction served"]

slow_events = [e for e in served_events if e["duration_ms"] > 100]
T3_slow_fraction = len(slow_events) / len(served_events)
T3_worst_id = max(served_events, key=lambda e: e["duration_ms"])["request_id"]

print("fraction over 100 ms:", T3_slow_fraction)
print("slowest request     :", T3_worst_id)

fraction over 100 ms: 0.21052631578947367
slowest request     : c456e123


In [15]:
# ------------------------------------------------------------------
# SELF-CHECK --- run this when you have attempted the tasks above.
# It never breaks your notebook. A task you have not done yet simply
# shows FAIL.
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    """Evaluate one graded condition without ever raising."""
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check('T1 | the logger threshold is WARNING', lambda: logger.level == logging.WARNING)
_check('T1 | T1_kept names exactly the three surviving levels', lambda: set(T1_kept) == {'WARNING', 'ERROR', 'CRITICAL'})
_check('T2 | the line has exactly the five required keys', lambda: set(T2_line) == {'ts', 'level', 'service', 'msg', 'env'})
_check('T2 | service and env carry the fixed values', lambda: T2_line['service'] == 'delivery-api' and T2_line['env'] == 'lab')
_check('T2 | level and message come from the record', lambda: T2_line['level'] == 'INFO' and T2_line['msg'] == 'prediction served')
_check('T2 | it still works for a different record', lambda: (lambda d: d['level'] == 'ERROR' and d['msg'] == 'boom')(json.loads(ServiceFormatter().format(logger.makeRecord('delivery', logging.ERROR, 'x', 1, 'boom', None, None)))))
_check('T3 | T3_slow_fraction matches the log file', lambda: abs(float(T3_slow_fraction) - (len([e for e in events if e.get('message') == 'prediction served' and e['duration_ms'] > 100]) / len([e for e in events if e.get('message') == 'prediction served']))) < 1e-9)
_check('T3 | T3_worst_id is the slowest served request', lambda: T3_worst_id == max([e for e in events if e.get('message') == 'prediction served'], key=lambda e: e['duration_ms'])['request_id'])

print("==================================================================")
print("SELF-CHECK   Practical 10 --- Logs You Can Actually Search")
print("==================================================================")
for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")
print("------------------------------------------------------------------")
_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("==================================================================")
if _passed == len(_results):
    print("Well done. Save the notebook and submit it.")
else:
    print("Read the FAIL lines above, fix those tasks, run this cell again.")

SELF-CHECK   Practical 10 --- Logs You Can Actually Search
  [PASS]  T1 | the logger threshold is WARNING
  [PASS]  T1 | T1_kept names exactly the three surviving levels
  [PASS]  T2 | the line has exactly the five required keys
  [PASS]  T2 | service and env carry the fixed values
  [PASS]  T2 | level and message come from the record
  [PASS]  T2 | it still works for a different record
  [PASS]  T3 | T3_slow_fraction matches the log file
  [PASS]  T3 | T3_worst_id is the slowest served request
------------------------------------------------------------------
  8 of 8 checks passed
Well done. Save the notebook and submit it.
